# Recruit to Roster Matcher

Match high school recruits to college football rosters using:
1. Name similarity matching (fuzzy)
2. Explicit college-to-recruit school mapping (deterministic)
3. Multi-layer backup strategies

## Setup: Import Libraries

In [ ]:
import pandas as pd
import os
import re
import time
import difflib
import json
from pathlib import Path

# Install fuzzywuzzy if not already installed
try:
    from fuzzywuzzy import fuzz, process
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', 'fuzzywuzzy'])
    from fuzzywuzzy import fuzz, process

print("Libraries loaded successfully")

# Set base directories for relative path resolution
# BASE_DATA_DIR: data folder inside Gridiron_Intelligence project
# BASE_PROJECT_DIR: parent of Gridiron_Intelligence (contains Football Recruitment Tables)
BASE_DATA_DIR = os.path.join(os.path.dirname(os.path.dirname(__file__)), "data")
BASE_PROJECT_DIR = os.path.dirname(os.path.dirname(os.path.dirname(__file__)))

# Fallback to relative paths if __file__ not available
if not os.path.exists(BASE_DATA_DIR):
    BASE_DATA_DIR = "data"
if not os.path.exists(BASE_PROJECT_DIR):
    BASE_PROJECT_DIR = ".."

print(f"Base data directory: {os.path.abspath(BASE_DATA_DIR)}")
print(f"Base project directory: {os.path.abspath(BASE_PROJECT_DIR)}")

# Load nickname mapping from JSON
nickname_map_path = os.path.join(BASE_DATA_DIR, "name_matching", "nickname_map.json")
with open(nickname_map_path, 'r') as f:
    nickname_data = json.load(f)
    nickname_map = nickname_data.get('nicknames', {})

print(f"✓ Loaded nickname map with {len(nickname_map)} entries")

Libraries loaded successfully
✓ Loaded nickname map with 359 entries


## Section 1: Load & Deduplicate Data

In [34]:
def clean_name(name):
    """
    Cleans player/team names for matching:
    - Removes special characters and apostrophes
    - Normalizes spacing
    - Converts to lowercase for comparison
    """
    if not name or pd.isna(name):
        return ""
    name = str(name).lower().strip()
    # Remove special characters except spaces and hyphens
    name = re.sub(r"[^a-z0-9\s\-]", "", name)
    # Normalize multiple spaces
    name = re.sub(r"\s+", " ", name).strip()
    return name

def split_name(name_clean):
    """
    Splits a cleaned name into first and last components.
    Returns tuple (first_name, last_name).
    """
    if not name_clean or len(name_clean.split()) == 0:
        return name_clean, ""
    
    parts = name_clean.split()
    if len(parts) == 1:
        return parts[0], ""
    else:
        return parts[0], ' '.join(parts[1:])

print("Name cleaning functions defined")

Name cleaning functions defined


In [ ]:
def load_recruits_data(recruits_folder=None):
    """
    Loads all recruiting data from CSV files in the recruitment folder.
    Deduplicates to keep FIRST record per player_id.
    
    Args:
        recruits_folder: Path to recruitment folder. If None, uses BASE_PROJECT_DIR/Football Reruitment Tables/Recruitment Classes
    """
    if recruits_folder is None:
        recruits_folder = os.path.join(BASE_PROJECT_DIR, "Football Reruitment Tables", "Recruitment Classes")
    
    print("Loading recruits data...")
    
    all_recruits = []
    
    if not os.path.exists(recruits_folder):
        print(f"ERROR: Recruits folder not found: {recruits_folder}")
        return pd.DataFrame()
    
    for file in sorted(os.listdir(recruits_folder)):
        if file.startswith('recruits_') and file.endswith('.csv'):
            file_path = os.path.join(recruits_folder, file)
            try:
                df = pd.read_csv(file_path)
                print(f"  {file}: {len(df)} records")
                all_recruits.append(df)
            except Exception as e:
                print(f"  ERROR loading {file}: {e}")
    
    if not all_recruits:
        print("No recruits data found!")
        return pd.DataFrame()
    
    recruits_df = pd.concat(all_recruits, ignore_index=True)
    print(f"\nTotal records before dedup: {len(recruits_df)}")
    
    # Deduplicate: keep FIRST record per player_id
    recruits_df = recruits_df.sort_values('year').drop_duplicates(subset=['player_id'], keep='first')
    print(f"Total records after dedup (first per player_id): {len(recruits_df)}")
    
    return recruits_df

def load_roster_data(rosters_dir=None):
    """
    Loads all roster CSV files and combines them.
    Deduplicates to keep FIRST record (earliest year) per Player_ID.
    
    Args:
        rosters_dir: Path to rosters directory. If None, uses BASE_DATA_DIR/Stats/rosters
    """
    if rosters_dir is None:
        rosters_dir = os.path.join(BASE_DATA_DIR, "Stats", "rosters")
    
    print("\nLoading roster data...")
    
    all_rosters = []
    
    if not os.path.exists(rosters_dir):
        print(f"ERROR: Rosters directory not found: {rosters_dir}")
        return pd.DataFrame()
    
    roster_files = sorted([f for f in os.listdir(rosters_dir) if f.endswith('_roster.csv')])
    print(f"Found {len(roster_files)} roster files")
    
    for file in roster_files:
        file_path = os.path.join(rosters_dir, file)
        try:
            df = pd.read_csv(file_path)
            keep_cols = ['Player', 'Player_ID', 'class', 'pos', 'summary', 'School', 'Year']
            available_cols = [c for c in keep_cols if c in df.columns]
            df = df[available_cols]
            all_rosters.append(df)
        except Exception as e:
            print(f"  ERROR loading {file}: {e}")
    
    if not all_rosters:
        print("No roster data found!")
        return pd.DataFrame()
    
    rosters_df = pd.concat(all_rosters, ignore_index=True)
    print(f"Total records before dedup: {len(rosters_df)}")
    
    # Deduplicate: keep FIRST record (earliest year) per Player_ID
    rosters_df = rosters_df.sort_values('Year').drop_duplicates(subset=['Player_ID'], keep='first')
    print(f"Total records after dedup (first per Player_ID): {len(rosters_df)}")
    
    return rosters_df

# LOAD DATA
print("="*80)
print("SECTION 1: LOAD & DEDUPLICATE DATA")
print("="*80)

recruits_df = load_recruits_data()
rosters_df = load_roster_data()

print(f"\nFinal data shapes:")
print(f"  Recruits: {recruits_df.shape}")
print(f"  Rosters:  {rosters_df.shape}")

SECTION 1: LOAD & DEDUPLICATE DATA
Loading recruits data...
  recruits_2015.csv: 3518 records
  recruits_2016.csv: 3942 records
  recruits_2017.csv: 4212 records
  recruits_2018.csv: 3886 records
  recruits_2019.csv: 3986 records
  recruits_2020.csv: 3862 records
  recruits_2021.csv: 2665 records
  recruits_2022.csv: 2194 records
  recruits_2023.csv: 2294 records
  recruits_2024.csv: 3652 records
  recruits_2025.csv: 3566 records
  recruits_2026.csv: 2933 records
  recruits_2027.csv: 989 records
  recruits_2028.csv: 227 records

Total records before dedup: 41926
Total records after dedup (first per player_id): 41926

Loading roster data...
Found 1440 roster files
Total records before dedup: 161593
Total records after dedup (first per Player_ID): 63097

Final data shapes:
  Recruits: (41926, 13)
  Rosters:  (63097, 7)


## Section 1B: Clean Names & Create Base Lists

In [ ]:
print("\n" + "="*80)
print("SECTION 1B: CLEAN NAMES & SAVE BASE LISTS")
print("="*80)

# Add cleaned names
recruits_df['name_cleaned'] = recruits_df['name'].apply(clean_name)
rosters_df['Player_cleaned'] = rosters_df['Player'].apply(clean_name)
rosters_df['School_cleaned'] = rosters_df['School'].apply(clean_name)

# Prepare export directory
output_dir = os.path.join(BASE_DATA_DIR, "name_matching")
os.makedirs(output_dir, exist_ok=True)

# Export cleaned recruits
recruits_export_cols = ['player_id', 'name', 'name_cleaned', 'year', 'committed_to', 'position', 'rating']
available_recruits_cols = [c for c in recruits_export_cols if c in recruits_df.columns]
recruits_export_df = recruits_df[available_recruits_cols].copy()

recruits_csv = os.path.join(output_dir, "all_recruits_cleaned.csv")
recruits_export_df.to_csv(recruits_csv, index=False)
print(f"\n✓ Exported {len(recruits_export_df)} cleaned recruits")
print(f"  File: all_recruits_cleaned.csv")
print(f"  Columns: {', '.join(available_recruits_cols)}")

# Export cleaned rosters
rosters_export_cols = ['Player_ID', 'Player', 'Player_cleaned', 'Year', 'School', 'School_cleaned', 'pos', 'class']
available_rosters_cols = [c for c in rosters_export_cols if c in rosters_df.columns]
rosters_export_df = rosters_df[available_rosters_cols].copy()

rosters_csv = os.path.join(output_dir, "all_rosters_cleaned.csv")
rosters_export_df.to_csv(rosters_csv, index=False)
print(f"\n✓ Exported {len(rosters_export_df)} cleaned rosters")
print(f"  File: all_rosters_cleaned.csv")
print(f"  Columns: {', '.join(available_rosters_cols)}")


SECTION 1B: CLEAN NAMES & SAVE BASE LISTS

✓ Exported 41926 cleaned recruits
  File: all_recruits_cleaned.csv
  Columns: player_id, name, name_cleaned, year, committed_to, position, rating

✓ Exported 63097 cleaned rosters
  File: all_rosters_cleaned.csv
  Columns: Player_ID, Player, Player_cleaned, Year, School, School_cleaned, pos, class


## Section 2: Identify D1 Colleges & Recruit Schools

In [37]:
print("\n" + "="*80)
print("SECTION 2: IDENTIFY D1 COLLEGES & RECRUIT COMMITMENT SCHOOLS")
print("="*80)

# D1 college list (from rosters - these are authoritative)
college_schools_list = sorted(rosters_df['School'].unique().tolist())

print(f"\nD1 College Schools (from rosters): {len(college_schools_list)}")
for i, school in enumerate(college_schools_list[:15], 1):
    print(f"  {i:2d}. {school}")
if len(college_schools_list) > 15:
    print(f"  ... and {len(college_schools_list) - 15} more")

# All recruit commitment schools (including D2, D3, etc.)
recruit_schools_list = sorted(recruits_df[recruits_df['committed_to'].notna()]['committed_to'].unique().tolist())
uncommitted_mask = recruits_df['committed_to'] == 'Uncommitted'

print(f"\nRecruit Commitment Schools (all divisions): {len(recruit_schools_list)}")
for i, school in enumerate(recruit_schools_list[:15], 1):
    print(f"  {i:2d}. {school}")
if len(recruit_schools_list) > 15:
    print(f"  ... and {len(recruit_schools_list) - 15} more")

# Uncommitted recruits
uncommitted_count = uncommitted_mask.sum()
print(f"\nUncommitted recruits: {uncommitted_count}")

# NOTE: D1 identification will be done AFTER loading college_to_recruit_mapping
# to correctly handle school variants (e.g., TCU -> Texas Christian)


SECTION 2: IDENTIFY D1 COLLEGES & RECRUIT COMMITMENT SCHOOLS

D1 College Schools (from rosters): 137
   1. Air Force
   2. Akron
   3. Alabama
   4. Appalachian State
   5. Arizona
   6. Arizona State
   7. Arkansas
   8. Arkansas State
   9. Army
  10. Auburn
  11. BYU
  12. Ball State
  13. Baylor
  14. Boise State
  15. Boston College
  ... and 122 more

Recruit Commitment Schools (all divisions): 394
   1. Abilene Christian
   2. Air Force
   3. Akron
   4. Alabama
   5. Alabama A&M
   6. Alabama State
   7. Albany
   8. Albany State
   9. Alcorn State
  10. Appalachian State
  11. Arizona
  12. Arizona State
  13. Arkansas
  14. Arkansas State
  15. Arkansas Tech
  ... and 379 more

Uncommitted recruits: 6765


## Section 3: Create College-to-Recruit School Mapping Template

In [ ]:
'''print("\n" + "="*80)
print("SECTION 3: EXPORT COLLEGE LIST FOR SCHOOL MAPPING")
print("="*80)

# Create template: college → list of recruit schools that map to it
college_mapping_template = {college: [] for college in college_schools_list}

# Export as JSON
college_mapping_json_path = os.path.join(output_dir, "college_to_recruit_schools_mapping.json")
with open(college_mapping_json_path, 'w') as f:
    json.dump(college_mapping_template, f, indent=2)

print(f"\n✓ Exported college mapping template as JSON")
print(f"  File: college_to_recruit_schools_mapping.json")
print(f"  Colleges: {len(college_schools_list)}")
print(f"  Format: {{\"College Name\": [\"Recruit School 1\", \"Recruit School 2\", ...], ...}}")

# Export as CSV for easy viewing/editing
college_mapping_csv_path = os.path.join(output_dir, "college_to_recruit_schools_mapping.csv")
college_csv_df = pd.DataFrame({
    'college_school': college_schools_list,
    'recruit_school_aliases': [''] * len(college_schools_list)
})
college_csv_df.to_csv(college_mapping_csv_path, index=False)

#Export Recruit Schools to list
recruit_mapping_csv_path = os.path.join(output_dir, "recruit_schools_list.csv")
recruit_csv_df = pd.DataFrame({
    'recruit_school': recruit_schools_list
})
recruit_csv_df.to_csv(recruit_mapping_csv_path, index=False)

print(f"\n✓ Also exported as CSV")
print(f"  File: college_to_recruit_schools_mapping.csv")
print(f"  Use this to easily see/edit the mapping")

print(f"\n" + "="*80)
print("NEXT STEP: Edit college_to_recruit_schools_mapping.json")
print("Add recruit school names that should map to each college.")
print("Example: \"Ohio State\": [\"Ohio State University\", \"OSU\", \"The Ohio State University\"]")
print("="*80)'''

'print("\n" + "="*80)\nprint("SECTION 3: EXPORT COLLEGE LIST FOR SCHOOL MAPPING")\nprint("="*80)\n\n# Create template: college → list of recruit schools that map to it\ncollege_mapping_template = {college: [] for college in college_schools_list}\n\n# Export as JSON\ncollege_mapping_json_path = os.path.join(output_dir, "college_to_recruit_schools_mapping.json")\nwith open(college_mapping_json_path, \'w\') as f:\n    json.dump(college_mapping_template, f, indent=2)\n\nprint(f"\n✓ Exported college mapping template as JSON")\nprint(f"  File: college_to_recruit_schools_mapping.json")\nprint(f"  Colleges: {len(college_schools_list)}")\nprint(f"  Format: {{"College Name": ["Recruit School 1", "Recruit School 2", ...], ...}}")\n\n# Export as CSV for easy viewing/editing\ncollege_mapping_csv_path = os.path.join(output_dir, "college_to_recruit_schools_mapping.csv")\ncollege_csv_df = pd.DataFrame({\n    \'college_school\': college_schools_list,\n    \'recruit_school_aliases\': [\'\'] * len(colleg

## Section 4: Load & Validate School Mapping

In [ ]:
print("\n" + "="*80)
print("SECTION 4: LOAD & VALIDATE SCHOOL MAPPING")
print("="*80)

# Load the mapping JSON (should be edited manually by user)
college_mapping_json_path = os.path.join(output_dir, "college_to_recruit_schools_mapping.json")

try:
    with open(college_mapping_json_path, 'r') as f:
        college_to_recruit_mapping = json.load(f)
    print(f"✓ Loaded college to recruit school mapping")
except Exception as e:
    print(f"ERROR loading mapping: {e}")
    college_to_recruit_mapping = {}

# Display mapping statistics
mapped_colleges = [c for c, schools in college_to_recruit_mapping.items() if len(schools) > 0]
total_mappings = sum(len(schools) for schools in college_to_recruit_mapping.values())

print(f"\nMapping Statistics:")
print(f"  Total colleges: {len(college_to_recruit_mapping)}")
print(f"  Colleges with mappings: {len(mapped_colleges)}")
print(f"  Total recruit school aliases: {total_mappings}")

if mapped_colleges:
    print(f"\nSample of populated mappings:")
    for i, college in enumerate(mapped_colleges[:5], 1):
        schools = college_to_recruit_mapping[college]
        print(f"  {college}:")
        for school in schools:
            print(f"    → {school}")

# =====================================================================================
# IDENTIFY D1 RECRUITS USING MAPPING (handles school variants like TCU -> Texas Christian)
# =====================================================================================
print(f"\n" + "="*80)
print("IDENTIFYING D1 VS NON-D1 RECRUITS (using college mapping)")
print("="*80)

# Build set of ALL recruit school variants that map to D1 colleges
all_d1_recruit_schools = set()
for college, recruit_variants in college_to_recruit_mapping.items():
    for variant in recruit_variants:
        all_d1_recruit_schools.add(variant)

print(f"\nD1 recruit school variants in mapping: {len(all_d1_recruit_schools)}")
if len(all_d1_recruit_schools) > 0:
    print(f"Sample: {sorted(list(all_d1_recruit_schools))[:20]}")

# Identify recruits: use mapping variants instead of just rosters
committed_to_d1_mask = recruits_df['committed_to'].isin(all_d1_recruit_schools)
d1_count = committed_to_d1_mask.sum()
print(f"\nRecruits committed to D1 schools (via mapping): {d1_count}")

# Recruits committed to non-D1 schools
non_d1_mask = (recruits_df['committed_to'].notna()) & (~uncommitted_mask) & (~committed_to_d1_mask)
non_d1_count = non_d1_mask.sum()
print(f"Recruits committed to non-D1 schools: {non_d1_count}")

print(f"\nSummary:")
print(f"  Total recruits: {len(recruits_df)}")
print(f"  → D1 committed: {d1_count} ({100*d1_count/len(recruits_df):.1f}%)")
print(f"  → Non-D1 committed: {non_d1_count} ({100*non_d1_count/len(recruits_df):.1f}%)")
print(f"  → Uncommitted: {uncommitted_count} ({100*uncommitted_count/len(recruits_df):.1f}%)")


SECTION 4: LOAD & VALIDATE SCHOOL MAPPING
✓ Loaded college to recruit school mapping

Mapping Statistics:
  Total colleges: 137
  Colleges with mappings: 137
  Total recruit school aliases: 137

Sample of populated mappings:
  Air Force:
    → Air Force
  Akron:
    → Akron
  Alabama:
    → Alabama
  Appalachian State:
    → Appalachian State
  Arizona:
    → Arizona

IDENTIFYING D1 VS NON-D1 RECRUITS (using college mapping)

D1 recruit school variants in mapping: 137
Sample: ['Air Force', 'Akron', 'Alabama', 'Appalachian State', 'Arizona', 'Arizona State', 'Arkansas', 'Arkansas State', 'Army', 'Auburn', 'BYU', 'Ball State', 'Baylor', 'Boise State', 'Boston College', 'Bowling Green', 'Buffalo', 'California', 'Central Michigan', 'Charlotte']

Recruits committed to D1 schools (via mapping): 30585
Recruits committed to non-D1 schools: 4576

Summary:
  Total recruits: 41926
  → D1 committed: 30585 (72.9%)
  → Non-D1 committed: 4576 (10.9%)
  → Uncommitted: 6765 (16.1%)


In [40]:
# =====================================================================================
# IDENTIFY D1 RECRUITS USING MAPPING (handles school variants like TCU -> Texas Christian)
# =====================================================================================
print(f"\n" + "="*80)
print("IDENTIFYING D1 VS NON-D1 RECRUITS (using college mapping)")
print("="*80)

# Build set of ALL recruit school variants that map to D1 colleges
all_d1_recruit_schools = set()
for college, recruit_variants in college_to_recruit_mapping.items():
    for variant in recruit_variants:
        all_d1_recruit_schools.add(variant)

print(f"\nD1 recruit school variants in mapping: {len(all_d1_recruit_schools)}")
if len(all_d1_recruit_schools) > 0:
    print(f"Sample: {sorted(list(all_d1_recruit_schools))[:20]}")

# Identify recruits: use mapping variants instead of just rosters
committed_to_d1_mask = recruits_df['committed_to'].isin(all_d1_recruit_schools)
d1_count = committed_to_d1_mask.sum()
print(f"\nRecruits committed to D1 schools (via mapping): {d1_count}")

# Recruits committed to non-D1 schools
non_d1_mask = (recruits_df['committed_to'].notna()) & (~uncommitted_mask) & (~committed_to_d1_mask)
non_d1_count = non_d1_mask.sum()
print(f"Recruits committed to non-D1 schools: {non_d1_count}")

print(f"\nSummary:")
print(f"  Total recruits: {len(recruits_df)}")
print(f"  → D1 committed: {d1_count} ({100*d1_count/len(recruits_df):.1f}%)")
print(f"  → Non-D1 committed: {non_d1_count} ({100*non_d1_count/len(recruits_df):.1f}%)")
print(f"  → Uncommitted: {uncommitted_count} ({100*uncommitted_count/len(recruits_df):.1f}%)")


IDENTIFYING D1 VS NON-D1 RECRUITS (using college mapping)

D1 recruit school variants in mapping: 137
Sample: ['Air Force', 'Akron', 'Alabama', 'Appalachian State', 'Arizona', 'Arizona State', 'Arkansas', 'Arkansas State', 'Army', 'Auburn', 'BYU', 'Ball State', 'Baylor', 'Boise State', 'Boston College', 'Bowling Green', 'Buffalo', 'California', 'Central Michigan', 'Charlotte']

Recruits committed to D1 schools (via mapping): 30585
Recruits committed to non-D1 schools: 4576

Summary:
  Total recruits: 41926
  → D1 committed: 30585 (72.9%)
  → Non-D1 committed: 4576 (10.9%)
  → Uncommitted: 6765 (16.1%)


## Section 5: Pre-Matching Visibility - Player Lists

In [18]:
print("\n" + "="*80)
print("SECTION 5: PRE-MATCHING VISIBILITY")
print("="*80)

# Filter recruits: only those committed to D1 schools
recruits_d1 = recruits_df[committed_to_d1_mask].copy()

print(f"\n--- D1-COMMITTED RECRUITS (for PRIMARY matching) ---")
print(f"Count: {len(recruits_d1)}")
print(f"\nSample (first 20):")

display_recruits_cols = ['player_id', 'name_cleaned', 'year', 'committed_to']
available_display_cols = [c for c in display_recruits_cols if c in recruits_d1.columns]
print(recruits_d1[available_display_cols].head(20).to_string(index=False))

print(f"\n--- D1 COLLEGE ROSTER PLAYERS (first record per player) ---")
print(f"Count: {len(rosters_df)}")
print(f"\nSample (first 20):")

display_rosters_cols = ['Player_ID', 'Player_cleaned', 'School', 'Year', 'pos']
available_display_cols = [c for c in display_rosters_cols if c in rosters_df.columns]
print(rosters_df[available_display_cols].head(20).to_string(index=False))

print(f"\n--- SCHOOL MAPPING SUMMARY ---")
print(f"Colleges with recruit school mappings: {len(mapped_colleges)} / {len(college_schools_list)}")
if mapped_colleges:
    print(f"\nColleges and their recruit school aliases:")
    for college in sorted(mapped_colleges):
        schools = college_to_recruit_mapping[college]
        print(f"  {college}: {schools}")


SECTION 5: PRE-MATCHING VISIBILITY

--- D1-COMMITTED RECRUITS (for PRIMARY matching) ---
Count: 28486

Sample (first 20):
 player_id         name_cleaned  year  committed_to
 201500001     trenton thompson  2015       Georgia
 201500002          martez ivey  2015       Florida
 201500003         byron cowart  2015        Auburn
 201500004        iman marshall  2015           USC
 201500005         derwin james  2015 Florida State
 201500006      kahlil mckenzie  2015     Tennessee
 201500007       cece jefferson  2015       Florida
 201500008           josh sweat  2015 Florida State
 201500009     kevin toliver ii  2015           LSU
 201500010      malik jefferson  2015         Texas
 201500011           josh rosen  2015          UCLA
 201500012        calvin ridley  2015       Alabama
 201500013     terry beckner jr  2015      Missouri
 201500014          daylon mack  2015     Texas A&M
 201500015    tarvarus mcfadden  2015 Florida State
 201500016 keisean lucier-south  2015        

## Section 6A: Matching Configuration & Helper Functions

In [19]:
# =====================================================================================
# MATCHING CONFIGURATION & HELPER FUNCTIONS
# =====================================================================================

MATCHING_CONFIG = {
    'PRIMARY_LAYER': {
        'name_threshold': 0.90,      # Min name similarity for primary matcher
        'min_component': 0.85,        # Min per-component score
        'year_tolerance': 2           # Years after recruitment to search
    },
    'BACKUP_LAYER': {
        'name_threshold': 0.85,       # More relaxed for backup
        'year_tolerance': 3
    },
    'FINAL_LAYER': {
        'name_threshold': 0.90,       # Keep strict for uncommitted (90%)
        'year_tolerance': 3           # 3 years tolerance
    }
}

def match_name_components(recruit_first, recruit_last, player_first, player_last):
    """
    Matches name components separately using difflib SequenceMatcher.
    Returns average of first_name and last_name similarity ratios.
    """
    first_ratio = difflib.SequenceMatcher(None, recruit_first, player_first).ratio()
    
    if not recruit_last or not player_last:
        return first_ratio
    
    last_ratio = difflib.SequenceMatcher(None, recruit_last, player_last).ratio()
    return (first_ratio + last_ratio) / 2.0

def build_roster_index(rosters_df):
    """
    Build efficient bucketing index from roster data.
    Uses (Year, First_Letter_Of_Name) as key for fast candidate selection.
    """
    roster_index = {}
    
    for idx, player in rosters_df.iterrows():
        try:
            year = int(player['Year'])
            name = player['Player_cleaned']
            
            if not name:
                continue
            
            key = (year, name[0])
            if key not in roster_index:
                roster_index[key] = []
            
            roster_index[key].append(player)
        except (ValueError, TypeError):
            continue
    
    return roster_index

# Build roster index for efficient matching
print("\n" + "="*80)
print("SECTION 6A: MATCHING CONFIGURATION & BUILD ROSTER INDEX")
print("="*80)
roster_index = build_roster_index(rosters_df)
print(f"✓ Roster index built: {len(roster_index)} buckets covering {len(rosters_df)} players")
print(f"✓ Configuration loaded: PRIMARY, BACKUP, FINAL")


SECTION 6A: MATCHING CONFIGURATION & BUILD ROSTER INDEX
✓ Roster index built: 284 buckets covering 63097 players
✓ Configuration loaded: PRIMARY, BACKUP, FINAL


## Section 6: Primary Matcher (D1-Committed Only)

In [22]:
def primary_matcher(recruits_df, rosters_df, roster_index, college_to_recruit_mapping):
    """
    PRIMARY LAYER: Match D1-committed recruits to rosters using explicit school mapping.
    Criteria: 90%+ name similarity + explicit school match + year alignment (0-2 years)
    WITH DETAILED DIAGNOSTICS
    """
    config = MATCHING_CONFIG['PRIMARY_LAYER']
    matches = []
    unmatched = []
    removed_player_ids = set()
    
    # Diagnostic counters
    no_candidates_count = 0
    name_below_threshold_count = 0
    school_mismatch_count = 0
    name_scores_collected = []
    
    print(f"\n{'PROGRESS: PRIMARY LAYER - Starting':-^80}")
    print(f"\nProcessing {len(recruits_df)} D1-committed recruits...")
    print(f"Roster index buckets: {len(roster_index)}")
    print(f"School mapping coverage: {len([c for c, s in college_to_recruit_mapping.items() if len(s) > 0])} / {len(college_to_recruit_mapping)} colleges\n")
    
    progress_counter = 0
    for idx, recruit in recruits_df.iterrows():
        progress_counter += 1
        if progress_counter % 100 == 0:
            print(f"  Progress: {progress_counter}/{len(recruits_df)} recruits processed...")
        recruit_name = recruit.get('name', '')
        recruit_year = recruit.get('year')
        recruit_school = recruit.get('committed_to', '')
        recruit_id = recruit.get('player_id', '')
        recruit_name_clean = recruit.get('name_cleaned', '')
        
        if not recruit_name_clean or pd.isna(recruit_year):
            continue
        
        try:
            recruit_year = int(recruit_year)
        except (ValueError, TypeError):
            continue
        
        recruit_first, recruit_last = split_name(recruit_name_clean)
        
        # Find candidates by bucketing
        candidates = []
        first_letter = recruit_name_clean[0]
        
        for year_offset in range(config['year_tolerance'] + 1):
            target_year = recruit_year + year_offset
            key = (target_year, first_letter)
            if key in roster_index:
                candidates.extend(roster_index[key])
        
        if not candidates:
            no_candidates_count += 1
            unmatched.append({
                'recruit_name': recruit_name,
                'recruit_year': recruit_year,
                'committed_to': recruit_school,
                'recruit_id': recruit_id,
                'match_score': 0,
                'reason': 'no_candidates'
            })
            if no_candidates_count <= 5:
                print(f"  DEBUG [no_candidates #{no_candidates_count}]: {recruit_name} (year {recruit_year}) → {recruit_school}, no roster candidates found")
            continue
        
        # Find best name match
        best_match = None
        best_name_score = 0
        found_match = False
        
        for candidate in candidates:
            cand_first, cand_last = split_name(candidate['Player_cleaned'])
            component_score = match_name_components(recruit_first, recruit_last, cand_first, cand_last)
            
            if component_score >= config['min_component'] and component_score > best_name_score:
                best_name_score = component_score
                best_match = candidate
                found_match = True
        
        if not found_match or best_name_score < config['name_threshold']:
            name_below_threshold_count += 1
            if found_match:
                name_scores_collected.append(best_name_score)
                if name_below_threshold_count <= 5:
                    print(f"  DEBUG [name_below #{name_below_threshold_count}]: {recruit_name} (year {recruit_year}) → best match '{best_match['Player']}' score {best_name_score*100:.1f}% < {config['name_threshold']*100:.0f}%")
            unmatched.append({
                'recruit_name': recruit_name,
                'recruit_year': recruit_year,
                'committed_to': recruit_school,
                'recruit_id': recruit_id,
                'match_score': int(best_name_score * 100) if found_match else 0,
                'reason': 'name_below_threshold'
            })
            continue
        
        name_score = best_name_score
        roster_school = best_match['School']
        
        # Check explicit school mapping
        # Structure: college (key) -> [recruit variants] (values)
        # Match if: roster_school (D1 college) is a key AND recruit_school is in its variants
        school_match = False
        if roster_school in college_to_recruit_mapping:
            if recruit_school in college_to_recruit_mapping[roster_school]:
                school_match = True
        
        if school_match:
            year_diff = int(best_match['Year']) - recruit_year
            matches.append({
                'recruit_name': recruit_name,
                'recruit_year': recruit_year,
                'committed_to': recruit_school,
                'recruit_id': recruit_id,
                'sports_ref_id': best_match['Player_ID'],
                'roster_name': best_match['Player'],
                'roster_school': roster_school,
                'roster_year': int(best_match['Year']),
                'pos': best_match.get('pos', ''),
                'name_score': int(name_score * 100),
                'team_match': 1 if school_match else 0,
                'year_diff': year_diff,
                'match_strategy': 'PRIMARY_LAYER'
            })
            removed_player_ids.add(best_match['Player_ID'])
        else:
            school_mismatch_count += 1
            name_scores_collected.append(name_score)
            unmatched.append({
                'recruit_name': recruit_name,
                'recruit_year': recruit_year,
                'committed_to': recruit_school,
                'recruit_id': recruit_id,
                'match_score': int(name_score * 100),
                'reason': 'school_mismatch'
            })
    
    # Print diagnostic summary
    print(f"\n{'DIAGNOSTIC SUMMARY':-^80}")
    print(f"  Total processed: {len(recruits_df)}")
    print(f"  No roster candidates: {no_candidates_count} ({100*no_candidates_count/len(recruits_df):.1f}%)")
    print(f"  Name below {config['name_threshold']*100:.0f}% threshold: {name_below_threshold_count} ({100*name_below_threshold_count/len(recruits_df):.1f}%)")
    print(f"  School mapping mismatch: {school_mismatch_count} ({100*school_mismatch_count/len(recruits_df):.1f}%)")
    if name_scores_collected:
        print(f"  Avg name score (rejected candidates): {sum(name_scores_collected)/len(name_scores_collected)*100:.1f}%")
    
    return pd.DataFrame(matches), pd.DataFrame(unmatched), removed_player_ids

In [23]:
# =====================================================================================
# EXECUTE PRIMARY LAYER & EXPORT
# =====================================================================================

primary_matches_df, primary_unmatched_df, primary_removed_ids = primary_matcher(
    recruits_d1, rosters_df, roster_index, college_to_recruit_mapping
)

# Export PRIMARY results
if len(primary_matches_df) > 0:
    primary_file = os.path.join(output_dir, "matches_primary_layer.csv")
    primary_matches_df.to_csv(primary_file, index=False)
    print(f"\nExported: matches_primary_layer.csv ({len(primary_matches_df)} matches)")



-----------------------PROGRESS: PRIMARY LAYER - Starting-----------------------

Processing 28486 D1-committed recruits...
Roster index buckets: 284
School mapping coverage: 137 / 137 colleges

  DEBUG [name_below #1]: Ronald Jones II (year 2015) → best match 'Ronald Jones' score 88.5% < 90%
  Progress: 100/28486 recruits processed...
  Progress: 200/28486 recruits processed...
  Progress: 300/28486 recruits processed...
  Progress: 400/28486 recruits processed...
  Progress: 500/28486 recruits processed...
  Progress: 600/28486 recruits processed...
  Progress: 700/28486 recruits processed...
  Progress: 800/28486 recruits processed...
  Progress: 900/28486 recruits processed...
  Progress: 1000/28486 recruits processed...
  Progress: 1100/28486 recruits processed...
  Progress: 1200/28486 recruits processed...
  Progress: 1300/28486 recruits processed...
  Progress: 1400/28486 recruits processed...
  Progress: 1500/28486 recruits processed...
  Progress: 1600/28486 recruits process

In [41]:
# =====================================================================================
# RERUN PRIMARY MATCHER ON NEWLY IDENTIFIED D1 RECRUITS
# =====================================================================================

print("\n" + "="*80)
print("RERUN PRIMARY MATCHER ON NEWLY IDENTIFIED D1 RECRUITS")
print("="*80)

# Load previously matched recruits
primary_matches_previous = pd.read_csv(primary_file)
previously_matched_ids = set(primary_matches_previous['recruit_id'].unique())

print(f"\nPreviously matched recruits: {len(previously_matched_ids)}")

# Identify newly identified D1 recruits (not in previous matches)
recruits_d1_all = recruits_df[committed_to_d1_mask].copy()
recruits_d1_new = recruits_d1_all[~recruits_d1_all['player_id'].isin(previously_matched_ids)].copy()

print(f"Total D1 recruits (current identification): {len(recruits_d1_all)}")
print(f"Newly identified D1 recruits not in previous matches: {len(recruits_d1_new)}")
print(f"These {len(recruits_d1_new)} new recruits will be matched with PRIMARY_LAYER...\n")

# Run primary matcher on new recruits only
new_primary_matches_df, new_primary_unmatched_df, new_primary_removed_ids = primary_matcher(
    recruits_d1_new, rosters_df, roster_index, college_to_recruit_mapping
)

print(f"\n{'RESULTS FROM NEW RECRUITS':-^80}")
print(f"New primary matches: {len(new_primary_matches_df)}")
print(f"New primary unmatched: {len(new_primary_unmatched_df)}")

# Append new matches to existing primary_matches_primary_layer.csv
if len(new_primary_matches_df) > 0:
    combined_primary = pd.concat([primary_matches_previous, new_primary_matches_df], ignore_index=True)
    combined_primary.to_csv(primary_file, index=False)
    print(f"\n✓ Updated matches_primary_layer.csv")
    print(f"  Previous: {len(primary_matches_previous)}")
    print(f"  New: {len(new_primary_matches_df)}")
    print(f"  Total: {len(combined_primary)}")
else:
    print(f"\nNo new matches found from newly identified recruits")



RERUN PRIMARY MATCHER ON NEWLY IDENTIFIED D1 RECRUITS

Previously matched recruits: 22449
Total D1 recruits (current identification): 30585
Newly identified D1 recruits not in previous matches: 8136
These 8136 new recruits will be matched with PRIMARY_LAYER...


-----------------------PROGRESS: PRIMARY LAYER - Starting-----------------------

Processing 8136 D1-committed recruits...
Roster index buckets: 284
School mapping coverage: 137 / 137 colleges

  DEBUG [name_below #1]: Ronald Jones II (year 2015) → best match 'Ronald Jones' score 88.5% < 90%
  Progress: 100/8136 recruits processed...
  Progress: 200/8136 recruits processed...
  Progress: 300/8136 recruits processed...
  Progress: 400/8136 recruits processed...
  Progress: 500/8136 recruits processed...
  Progress: 600/8136 recruits processed...
  Progress: 700/8136 recruits processed...
  Progress: 800/8136 recruits processed...
  Progress: 900/8136 recruits processed...
  Progress: 1000/8136 recruits processed...
  Progress: 

## Section 6B: Backup Layer (D1-Committed, Name Variants)

In [24]:
# =====================================================================================
# NAME VARIANT HELPER FUNCTIONS
# =====================================================================================

def get_name_variants(name_clean):
    """
    Generate name variants: original + common nicknames from nickname_map.json.
    Returns list of name variants to try.
    """
    variants = [name_clean]  # Always include original
    
    parts = name_clean.split()
    if len(parts) > 0:
        first = parts[0].lower()
        # Check if first name has variants in loaded nickname map
        if first in nickname_map:
            # Add all variants from the nickname map
            variants_list = nickname_map[first]
            if isinstance(variants_list, list):
                for variant in variants_list:
                    nickname_variant = variant + (' ' + ' '.join(parts[1:]) if len(parts) > 1 else '')
                    if nickname_variant not in variants:  # Avoid duplicates
                        variants.append(nickname_variant)
            else:
                # Single variant
                nickname_variant = variants_list + (' ' + ' '.join(parts[1:]) if len(parts) > 1 else '')
                if nickname_variant not in variants:
                    variants.append(nickname_variant)
    
    return variants

def remove_suffixes(name_clean):
    """
    Remove generational suffixes (Jr, Sr, III, II, IV, V, etc.) from name.
    Returns name without suffix.
    """
    suffixes = ['jr', 'sr', 'iii', 'ii', 'iv', 'v', 'vi']
    parts = name_clean.split()
    
    filtered_parts = []
    for part in parts:
        if part.lower() not in suffixes:
            filtered_parts.append(part)
    
    return ' '.join(filtered_parts) if filtered_parts else name_clean

def get_initial_variants(name_clean):
    """
    Generate name with and without periods on initials.
    E.g., 'J Smith' -> ['J Smith', 'J. Smith'], 'J. Smith' -> ['J. Smith', 'J Smith']
    Returns list of variants.
    """
    variants = [name_clean]
    
    parts = name_clean.split()
    # Check if first part is initial (single letter optionally with period)
    if len(parts) > 0:
        first_part = parts[0]
        if len(first_part) == 1:  # Single letter without period
            variant_with_period = first_part + '.' + (' ' + ' '.join(parts[1:]) if len(parts) > 1 else '')
            variants.append(variant_with_period.strip())
        elif len(first_part) == 2 and first_part[1] == '.':
            # Single letter with period, try without
            variant_without_period = first_part[0] + (' ' + ' '.join(parts[1:]) if len(parts) > 1 else '')
            variants.append(variant_without_period.strip())
    
    return variants

def split_hyphenated(name_clean):
    """
    Split hyphenated names into components.
    E.g., 'smith-jones' -> ['smith-jones', 'smith', 'jones']
    Returns list of name components to try.
    """
    if '-' not in name_clean:
        return [name_clean]
    
    parts = name_clean.split()
    variants = [name_clean]  # Include original
    
    # Find hyphenated component (usually last name)
    for part in parts:
        if '-' in part:
            components = part.split('-')
            for comp in components:
                if comp.strip():
                    # Replace hyphenated part with component, keeping rest of name
                    variant = name_clean.replace(part, comp)
                    variants.append(variant.strip())
    
    return variants

print("✓ Name variant helper functions defined")


✓ Name variant helper functions defined


In [26]:
def backup_matcher(unmatched_df, rosters_df, roster_index, college_to_recruit_mapping, removed_ids):
    """
    BACKUP LAYER: 5 sequential name variant strategies for D1-committed recruits.
    Each strategy relaxes name matching (original → nicknames, suffixes, initials, hyphenated).
    All require: explicit school mapping + 0-2 year tolerance.
    """
    config = MATCHING_CONFIG['BACKUP_LAYER']
    all_matches = []
    remaining_unmatched = list(unmatched_df.to_dict('records'))
    
    # Remove already-matched players from pool
    rosters_available = rosters_df[~rosters_df['Player_ID'].isin(removed_ids)].copy()
    roster_index_backup = build_roster_index(rosters_available)
    
    print(f"\n{'PROGRESS: BACKUP LAYER - 5 Sequential Strategies':-^80}")
    print(f"\nInput: {len(unmatched_df)} unmatched D1 recruits")
    print(f"Available rosters: {len(rosters_available)}\n")
    
    # Strategy tracking
    strategies = [
        ('Original', lambda name: [name]),
        ('Nicknames', get_name_variants),
        ('Remove Suffixes', lambda name: [remove_suffixes(name)]),
        ('Initial Variants', get_initial_variants),
        ('Hyphenated Split', split_hyphenated)
    ]
    
    for strategy_num, (strategy_name, variant_fn) in enumerate(strategies, 1):
        if not remaining_unmatched:
            break
        
        matches_this_strategy = []
        next_remaining = []
        
        print(f"STRATEGY {strategy_num}/{len(strategies)}: {strategy_name}")
        print(f"  Processing {len(remaining_unmatched)} recruits...")
        
        strategy_progress_counter = 0
        for recruit in remaining_unmatched:
            strategy_progress_counter += 1
            if strategy_progress_counter % 100 == 0:
                print(f"    Progress: {strategy_progress_counter}/{len(remaining_unmatched)} recruits processed...")
            recruit_name = recruit.get('recruit_name', '')
            recruit_year = recruit.get('recruit_year')
            recruit_school = recruit.get('committed_to', '')
            recruit_id = recruit.get('recruit_id', '')
            
            if not recruit_name or pd.isna(recruit_year):
                next_remaining.append(recruit)
                continue
            
            try:
                recruit_year = int(recruit_year)
            except (ValueError, TypeError):
                next_remaining.append(recruit)
                continue
            
            # Try name variants
            name_variants = variant_fn(clean_name(recruit_name))
            found_match = False
            best_match = None
            best_name_score = 0
            
            for variant in name_variants:
                if not variant:
                    continue
                
                # Find candidates
                candidates = []
                first_letter = variant[0] if variant else ''
                
                for year_offset in range(config['year_tolerance'] + 1):
                    target_year = recruit_year + year_offset
                    key = (target_year, first_letter)
                    if key in roster_index_backup:
                        candidates.extend(roster_index_backup[key])
                
                # Try to match
                for candidate in candidates:
                    name_score = difflib.SequenceMatcher(None, variant.lower(), 
                                                         candidate['Player_cleaned'].lower()).ratio()
                    
                    if name_score >= 0.90 and name_score > best_name_score:
                        best_name_score = name_score
                        best_match = candidate
                        found_match = True
                
                if found_match:
                    break
            
            if not found_match:
                next_remaining.append(recruit)
                continue
            
            roster_school = best_match['School']
            
            # Check explicit school mapping
            # Structure: college (key) -> [recruit variants] (values)
            # Match if: roster_school (D1 college) is a key AND recruit_school is in its variants
            school_match = False
            if roster_school in college_to_recruit_mapping:
                if recruit_school in college_to_recruit_mapping[roster_school]:
                    school_match = True
            
            if school_match:
                year_diff = int(best_match['Year']) - recruit_year
                matches_this_strategy.append({
                    'recruit_name': recruit_name,
                    'recruit_year': recruit_year,
                    'committed_to': recruit_school,
                    'recruit_id': recruit_id,
                    'sports_ref_id': best_match['Player_ID'],
                    'roster_name': best_match['Player'],
                    'roster_school': roster_school,
                    'roster_year': int(best_match['Year']),
                    'pos': best_match.get('pos', ''),
                    'name_score': int(best_name_score * 100),
                    'team_match': 1,
                    'year_diff': year_diff,
                    'match_strategy': f'Backup_Strategy{strategy_num}_{strategy_name}'
                })
                removed_ids.add(best_match['Player_ID'])
            else:
                next_remaining.append(recruit)
        
        all_matches.extend(matches_this_strategy)
        remaining_unmatched = next_remaining
        print(f"  ✓ Found {len(matches_this_strategy)} matches (Remaining: {len(remaining_unmatched)})\n")
    
    return pd.DataFrame(all_matches), pd.DataFrame(remaining_unmatched), removed_ids

In [27]:
# =====================================================================================
# EXECUTE BACKUP LAYER & EXPORT
# =====================================================================================

backup_matches_df, backup_unmatched_df, backup_removed_ids = backup_matcher(
    primary_unmatched_df, rosters_df, roster_index, college_to_recruit_mapping, primary_removed_ids
)

# Export BACKUP results
if len(backup_matches_df) > 0:
    backup_file = os.path.join(output_dir, "matches_backup_layer.csv")
    backup_matches_df.to_csv(backup_file, index=False)
    print(f"\nExported: matches_backup_layer.csv ({len(backup_matches_df)} matches)")



----------------PROGRESS: BACKUP LAYER - 5 Sequential Strategies----------------

Input: 6037 unmatched D1 recruits
Available rosters: 40663

STRATEGY 1/5: Original
  Processing 6037 recruits...
    Progress: 100/6037 recruits processed...
    Progress: 200/6037 recruits processed...
    Progress: 300/6037 recruits processed...
    Progress: 400/6037 recruits processed...
    Progress: 500/6037 recruits processed...
    Progress: 600/6037 recruits processed...
    Progress: 700/6037 recruits processed...
    Progress: 800/6037 recruits processed...
    Progress: 900/6037 recruits processed...
    Progress: 1000/6037 recruits processed...
    Progress: 1100/6037 recruits processed...
    Progress: 1200/6037 recruits processed...
    Progress: 1300/6037 recruits processed...
    Progress: 1400/6037 recruits processed...
    Progress: 1500/6037 recruits processed...
    Progress: 1600/6037 recruits processed...
    Progress: 1700/6037 recruits processed...
    Progress: 1800/6037 recruit

In [42]:
# =====================================================================================
# RERUN BACKUP LAYER ON NEWLY IDENTIFIED D1 RECRUITS
# =====================================================================================

print("\n" + "="*80)
print("RERUN BACKUP LAYER ON NEWLY IDENTIFIED D1 RECRUITS")
print("="*80)

# Load previously matched recruits from both primary and backup
backup_matches_previous = pd.read_csv(backup_file)
primary_matches_updated = pd.read_csv(primary_file)

all_previously_matched_ids = set(
    list(primary_matches_updated['recruit_id'].unique()) +
    list(backup_matches_previous['recruit_id'].unique())
)

print(f"\nPreviously matched recruits (primary + backup): {len(all_previously_matched_ids)}")
print(f"  Primary matches: {len(set(primary_matches_updated['recruit_id'].unique()))}")
print(f"  Backup matches: {len(set(backup_matches_previous['recruit_id'].unique()))}")

# Identify newly identified D1 recruits NOT in primary or backup matches
recruits_d1_all = recruits_df[committed_to_d1_mask].copy()
recruits_d1_unmatched = recruits_d1_all[~recruits_d1_all['player_id'].isin(all_previously_matched_ids)].copy()

print(f"\nTotal D1 recruits: {len(recruits_d1_all)}")
print(f"D1 recruits not in primary or backup: {len(recruits_d1_unmatched)}")
print(f"These {len(recruits_d1_unmatched)} recruits will be matched with BACKUP_LAYER...\n")

# Create unmatched dataframe for backup matcher (in the format it expects)
# Backup matcher expects: recruit_name, recruit_year, committed_to, recruit_id, match_score, reason
recruits_d1_unmatched_formatted = recruits_d1_unmatched[[
    'name', 'year', 'committed_to', 'player_id'
]].copy()
recruits_d1_unmatched_formatted.columns = ['recruit_name', 'recruit_year', 'committed_to', 'recruit_id']
recruits_d1_unmatched_formatted['match_score'] = 0
recruits_d1_unmatched_formatted['reason'] = 'not_yet_matched'

# Run backup matcher on new unmatched recruits
# Update removed_ids to include players already removed in primary layer
all_removed_ids = primary_removed_ids | backup_removed_ids | new_primary_removed_ids

new_backup_matches_df, new_backup_unmatched_df, new_backup_removed_ids = backup_matcher(
    recruits_d1_unmatched_formatted, rosters_df, roster_index, college_to_recruit_mapping, all_removed_ids
)

print(f"\n{'RESULTS FROM NEWLY IDENTIFIED D1 RECRUITS':-^80}")
print(f"New backup matches: {len(new_backup_matches_df)}")
print(f"New backup unmatched: {len(new_backup_unmatched_df)}")

# Append new matches to existing matches_backup_layer.csv
if len(new_backup_matches_df) > 0:
    combined_backup = pd.concat([backup_matches_previous, new_backup_matches_df], ignore_index=True)
    combined_backup.to_csv(backup_file, index=False)
    print(f"\n✓ Updated matches_backup_layer.csv")
    print(f"  Previous: {len(backup_matches_previous)}")
    print(f"  New: {len(new_backup_matches_df)}")
    print(f"  Total: {len(combined_backup)}")
else:
    print(f"\nNo new matches found from newly identified recruits")



RERUN BACKUP LAYER ON NEWLY IDENTIFIED D1 RECRUITS

Previously matched recruits (primary + backup): 24701
  Primary matches: 24122
  Backup matches: 579

Total D1 recruits: 30585
D1 recruits not in primary or backup: 5884
These 5884 recruits will be matched with BACKUP_LAYER...


----------------PROGRESS: BACKUP LAYER - 5 Sequential Strategies----------------

Input: 5884 unmatched D1 recruits
Available rosters: 38413

STRATEGY 1/5: Original
  Processing 5884 recruits...
    Progress: 100/5884 recruits processed...
    Progress: 200/5884 recruits processed...
    Progress: 300/5884 recruits processed...
    Progress: 400/5884 recruits processed...
    Progress: 500/5884 recruits processed...
    Progress: 600/5884 recruits processed...
    Progress: 700/5884 recruits processed...
    Progress: 800/5884 recruits processed...
    Progress: 900/5884 recruits processed...
    Progress: 1000/5884 recruits processed...
    Progress: 1100/5884 recruits processed...
    Progress: 1200/5884 re

## Section 7 (Final Layer - Positional Matching. Uncommited, D2/D3, First Initial)

In [43]:
print("\n" + "="*80)
print("SECTION 7: FINAL LAYER (Non-D1 Recruits)")
print("="*80)

# Filter: recruits NOT committed to D1 schools (D2, D3, Uncommitted)
recruits_non_d1 = recruits_df[~committed_to_d1_mask].copy()

print(f"\nNon-D1 Recruits for final layer: {len(recruits_non_d1)}")
print(f"  - Uncommitted: {(recruits_non_d1['committed_to'] == 'Uncommitted').sum()}")
print(f"  - D2/D3 committed: {((recruits_non_d1['committed_to'] != 'Uncommitted') & (recruits_non_d1['committed_to'].notna())).sum()}")

print(f"\nThese recruits will be matched using 2 strategies:")
print(f"  Strategy A: Standard name matching (90%+ fuzzy + ±3 years + position)")
print(f"  Strategy B: Initial + LastName exact (first_initial + exact_last_name + ±2 years + position + school)")

print(f"\nSample of non-D1 recruits:")
display_cols = ['player_id', 'name_cleaned', 'year', 'committed_to']
available_cols = [c for c in display_cols if c in recruits_non_d1.columns]
print(recruits_non_d1[available_cols].head(15).to_string(index=False))


SECTION 7: FINAL LAYER (Non-D1 Recruits)

Non-D1 Recruits for final layer: 11341
  - Uncommitted: 6765
  - D2/D3 committed: 4576

These recruits will be matched using 2 strategies:
  Strategy A: Standard name matching (90%+ fuzzy + ±3 years + position)
  Strategy B: Initial + LastName exact (first_initial + exact_last_name + ±2 years + position + school)

Sample of non-D1 recruits:
 player_id           name_cleaned  year       committed_to
 201500301           ahmad wagner  2015        Uncommitted
 201500448           markel smith  2015        Uncommitted
 201500538          justin bailey  2015        Uncommitted
 201500558             brian bell  2015        Uncommitted
 201500583       joseph bussireth  2015        Uncommitted
 201500642 justice shelton-mosley  2015            Harvard
 201500666    darquavious mangham  2015        Uncommitted
 201500674            tyrell reed  2015    Henderson State
 201500689         eddie rudinski  2015        Uncommitted
 201500696     dominique

## Section 7B: Final Layer (Non-D1: Name, Year, Position Only)

In [44]:
def final_matcher(recruits_non_d1, rosters_df, roster_index, removed_ids):
    """
    FINAL LAYER: Match non-D1 recruits using 2 strategies:
    Strategy A (Standard): 90%+ name + year tolerance + position match
    Strategy B (Initial+LastName): First initial + exact last name + 2-year tolerance + position + school match
    """
    config = MATCHING_CONFIG['FINAL_LAYER']
    matches = []
    remaining_unmatched = []
    
    # Remove already-matched players
    rosters_available = rosters_df[~rosters_df['Player_ID'].isin(removed_ids)].copy()
    roster_index_final = build_roster_index(rosters_available)
    
    print(f"\n{'PROGRESS: FINAL LAYER - Non-D1 Recruits (2 Strategies)':-^80}")
    print(f"\nProcessing {len(recruits_non_d1)} non-D1 recruits...")
    print(f"Available rosters: {len(rosters_available)}\n")
    
    # Position mapping for uncommitted
    position_similarity = {
        'qb': ['qb'],
        'rb': ['rb', 'hb'], 'hb': ['rb'],
        'wr': ['wr', 'receiver'],
        'te': ['te'],
        'ol': ['ol', 'ot', 'og', 'c'],
        'de': ['de', 'edge', 'dl'], 'edge': ['de', 'edge', 'dl'],
        'dt': ['dt', 'dl', 'nt'],
        'lb': ['lb', 'olb', 'ilb', 'mlb'],
        'cb': ['cb', 'db'],
        'db': ['db', 's', 'fs', 'ss'],
        'k': ['k'],
        'p': ['p'],
        'ls': ['ls'],
        'ath': ['db', 'rb', 'wr', 'dl']
    }
    
    # STRATEGY A: Standard full name matching (90%+ name + position + year tolerance)
    strategy_a_matches = []
    strategy_a_unmatched = []
    
    print("STRATEGY A: Standard Name + Year + Position")
    strategy_a_count = 0
    strategy_a_progress = 0
    
    for idx, recruit in recruits_non_d1.iterrows():
        strategy_a_progress += 1
        if strategy_a_progress % 100 == 0:
            print(f"  Progress: {strategy_a_progress}/{len(recruits_non_d1)} recruits processed...")
        recruit_name = recruit.get('name', '')
        recruit_year = recruit.get('year')
        recruit_id = recruit.get('player_id', '')
        recruit_pos = str(recruit.get('position', '')).lower().strip() if recruit.get('position') else ''
        recruit_school = recruit.get('committed_to', '')
        recruit_name_clean = recruit.get('name_cleaned', '')
        
        if not recruit_name_clean or pd.isna(recruit_year):
            strategy_a_unmatched.append({
                'recruit_name': recruit_name,
                'recruit_year': recruit_year,
                'committed_to': recruit_school,
                'recruit_id': recruit_id,
                'match_score': 0,
                'reason': 'no_name_or_year'
            })
            continue
        
        try:
            recruit_year = int(recruit_year)
        except (ValueError, TypeError):
            strategy_a_unmatched.append({
                'recruit_name': recruit_name,
                'recruit_year': recruit_year,
                'committed_to': recruit_school,
                'recruit_id': recruit_id,
                'match_score': 0,
                'reason': 'invalid_year'
            })
            continue
        
        recruit_first, recruit_last = split_name(recruit_name_clean)
        
        # Find candidates across all years within tolerance
        candidates_by_score = []
        first_letter = recruit_name_clean[0]
        
        for year_offset in range(config['year_tolerance'] + 1):
            target_year = recruit_year + year_offset
            key = (target_year, first_letter)
            if key not in roster_index_final:
                continue
            
            for player in roster_index_final[key]:
                name_match = difflib.SequenceMatcher(None, recruit_name_clean.lower(), 
                                                     player['Player_cleaned'].lower()).ratio()
                
                if name_match >= config['name_threshold']:
                    player_pos = str(player.get('pos', '')).lower().strip()
                    
                    # Check position compatibility
                    pos_match = False
                    if not recruit_pos:
                        pos_match = True  # No position data, allow
                    else:
                        similar_positions = position_similarity.get(recruit_pos, [])
                        pos_match = player_pos in similar_positions or recruit_pos == player_pos
                    
                    if pos_match:
                        candidates_by_score.append((name_match, player))
        
        if not candidates_by_score:
            strategy_a_unmatched.append({
                'recruit_name': recruit_name,
                'recruit_year': recruit_year,
                'committed_to': recruit_school,
                'recruit_id': recruit_id,
                'match_score': 0,
                'reason': 'no_candidates'
            })
            continue
        
        # Sort by score and take best match if clear winner (5%+ margin)
        candidates_by_score.sort(key=lambda x: x[0], reverse=True)
        best_score, best_player = candidates_by_score[0]
        
        if len(candidates_by_score) > 1:
            second_score = candidates_by_score[1][0]
            # Only match if clear winner
            if (best_score - second_score) < 0.05:
                strategy_a_unmatched.append({
                    'recruit_name': recruit_name,
                    'recruit_year': recruit_year,
                    'committed_to': recruit_school,
                    'recruit_id': recruit_id,
                    'match_score': int(best_score * 100),
                    'reason': 'ambiguous_match'
                })
                continue
        
        # Make match
        year_diff = int(best_player['Year']) - recruit_year
        strategy_a_matches.append({
            'recruit_name': recruit_name,
            'recruit_year': recruit_year,
            'committed_to': recruit_school,
            'recruit_id': recruit_id,
            'sports_ref_id': best_player['Player_ID'],
            'roster_name': best_player['Player'],
            'roster_school': best_player['School'],
            'roster_year': int(best_player['Year']),
            'pos': best_player.get('pos', ''),
            'recruit_pos': recruit_pos,
            'name_score': int(best_score * 100),
            'team_match': 0,
            'year_diff': year_diff,
            'match_strategy': 'Final_StandardName'
        })
        removed_ids.add(best_player['Player_ID'])
        strategy_a_count += 1
    
    matches.extend(strategy_a_matches)
    print(f"  ✓ Found {len(strategy_a_matches)} matches\n")
    
    # STRATEGY B: Initial + Last Name Exact Match (first initial + exact last name + 2yr tolerance + position + school match)
    strategy_b_matches = []
    strategy_b_unmatched = []
    
    print("STRATEGY B: Initial + LastName Exact (2-year tolerance + position + school match)")
    strategy_b_progress = 0
    
    for unmatched_recruit in strategy_a_unmatched:
        strategy_b_progress += 1
        if strategy_b_progress % 100 == 0:
            print(f"  Progress: {strategy_b_progress}/{len(strategy_a_unmatched)} recruits processed...")
        recruit_name = unmatched_recruit.get('recruit_name', '')
        recruit_year = unmatched_recruit.get('recruit_year')
        recruit_id = unmatched_recruit.get('recruit_id', '')
        recruit_school = unmatched_recruit.get('committed_to', '')
        
        # Get position from recruits_non_d1
        recruit_record = recruits_non_d1[recruits_non_d1['player_id'] == recruit_id]
        recruit_pos = str(recruit_record['position'].values[0]).lower().strip() if len(recruit_record) > 0 and recruit_record['position'].values[0] else ''
        
        if not recruit_name or pd.isna(recruit_year):
            strategy_b_unmatched.append(unmatched_recruit)
            continue
        
        try:
            recruit_year = int(recruit_year)
        except (ValueError, TypeError):
            strategy_b_unmatched.append(unmatched_recruit)
            continue
        
        recruit_name_clean = clean_name(recruit_name)
        if not recruit_name_clean:
            strategy_b_unmatched.append(unmatched_recruit)
            continue
        
        # Extract first initial and last name (exact)
        parts = recruit_name_clean.split()
        if len(parts) < 2:
            strategy_b_unmatched.append(unmatched_recruit)
            continue
        
        recruit_first_initial = parts[0][0]
        recruit_last_name = ' '.join(parts[1:])
        
        # Search for matches with same first initial + exact last name within 2-year range
        candidates = []
        
        for year_offset in range(-2, 3):
            target_year = recruit_year + year_offset
            for first_letter in [recruit_first_initial, recruit_first_initial.upper(), recruit_first_initial.lower()]:
                key = (target_year, first_letter)
                if key in roster_index_final:
                    for player in roster_index_final[key]:
                        player_name_clean = player['Player_cleaned']
                        player_parts = player_name_clean.split()
                        if len(player_parts) >= 2:
                            player_last_name = ' '.join(player_parts[1:])
                            # Exact last name match
                            if player_last_name == recruit_last_name:
                                player_pos = str(player.get('pos', '')).lower().strip()
                                # Check position match
                                pos_match = False
                                if not recruit_pos:
                                    pos_match = True
                                else:
                                    similar_positions = position_similarity.get(recruit_pos, [])
                                    pos_match = player_pos in similar_positions or recruit_pos == player_pos
                                
                                if pos_match:
                                    candidates.append((year_offset, player))
        
        # Only match if exactly 1 candidate found (avoid ambiguity)
        if len(candidates) == 1:
            year_offset, best_player = candidates[0]
            
            # Check school match using college mapping
            roster_school = best_player['School']
            
            # Use explicit school mapping if available (for non-D1 recruits that might have D1 alternatives)
            # Otherwise allow match if no school data for recruit
            school_match = False
            if recruit_school and roster_school in college_to_recruit_mapping:
                if recruit_school in college_to_recruit_mapping[roster_school]:
                    school_match = True
            
            if school_match or not recruit_school:  # Allow if no school data
                year_diff = int(best_player['Year']) - recruit_year
                strategy_b_matches.append({
                    'recruit_name': recruit_name,
                    'recruit_year': recruit_year,
                    'committed_to': recruit_school,
                    'recruit_id': recruit_id,
                    'sports_ref_id': best_player['Player_ID'],
                    'roster_name': best_player['Player'],
                    'roster_school': best_player['School'],
                    'roster_year': int(best_player['Year']),
                    'pos': best_player.get('pos', ''),
                    'recruit_pos': recruit_pos,
                    'name_score': 100,  # Exact last name match
                    'team_match': 1 if school_match else 0,
                    'year_diff': year_diff,
                    'match_strategy': 'Final_InitialLastName_Exact'
                })
                removed_ids.add(best_player['Player_ID'])
            else:
                strategy_b_unmatched.append(unmatched_recruit)
        else:
            strategy_b_unmatched.append(unmatched_recruit)
    
    matches.extend(strategy_b_matches)
    remaining_unmatched = strategy_b_unmatched
    print(f"  ✓ Found {len(strategy_b_matches)} matches\n")
    
    return pd.DataFrame(matches), pd.DataFrame(remaining_unmatched), removed_ids

In [ ]:
# =====================================================================================
# EXECUTE FINAL LAYER & EXPORT
# =====================================================================================

final_matches_df, final_unmatched_df, final_removed_ids = final_matcher(
    recruits_non_d1, rosters_df, roster_index, primary_removed_ids | backup_removed_ids
)

# Export FINAL results
if len(final_matches_df) > 0:
    final_file = os.path.join(output_dir, "matches_final_layer.csv")
    final_matches_df.to_csv(final_file, index=False)
    print(f"\nExported: matches_final_layer.csv ({len(final_matches_df)} matches)")

# Export UNMATCHED recruits
if len(final_unmatched_df) > 0:
    unmatched_file = os.path.join(output_dir, "unmatched_recruits.csv")
    final_unmatched_df.to_csv(unmatched_file, index=False)
    print(f"Exported: unmatched_recruits.csv ({len(final_unmatched_df)} unmatched)")



-------------PROGRESS: FINAL LAYER - Non-D1 Recruits (2 Strategies)-------------

Processing 11341 non-D1 recruits...
Available rosters: 40085

STRATEGY A: Standard Name + Year + Position
  Progress: 100/11341 recruits processed...
  Progress: 200/11341 recruits processed...
  Progress: 300/11341 recruits processed...
  Progress: 400/11341 recruits processed...
  Progress: 500/11341 recruits processed...
  Progress: 600/11341 recruits processed...
  Progress: 700/11341 recruits processed...
  Progress: 800/11341 recruits processed...
  Progress: 900/11341 recruits processed...
  Progress: 1000/11341 recruits processed...
  Progress: 1100/11341 recruits processed...
  Progress: 1200/11341 recruits processed...
  Progress: 1300/11341 recruits processed...
  Progress: 1400/11341 recruits processed...
  Progress: 1500/11341 recruits processed...
  Progress: 1600/11341 recruits processed...
  Progress: 1700/11341 recruits processed...
  Progress: 1800/11341 recruits processed...
  Progress

In [48]:
# Load previously matched recruits from all layers
primary_matches_final = pd.read_csv(primary_file)
backup_matches_final = pd.read_csv(backup_file)

# Combine all matches and export
all_matches = pd.concat([primary_matches_final, backup_matches_final, final_matches_df], ignore_index=True)
if len(all_matches) > 0:
    matches_file = os.path.join(output_dir, "all_matches_combined.csv")
    all_matches.to_csv(matches_file, index=False)
    print(f"Exported: all_matches_combined.csv ({len(all_matches)} total matches)")

print(f"\n" + "="*80)
print("MATCHING COMPLETE")
print("="*80)
print(f"PRIMARY:  {len(primary_matches_df)} matches")
print(f"BACKUP:   {len(backup_matches_df)} matches")
print(f"FINAL:    {len(final_matches_df)} matches")
print(f"TOTAL:    {len(all_matches)} matches")
print(f"UNMATCHED: {len(final_unmatched_df)}")

Exported: all_matches_combined.csv (25185 total matches)

MATCHING COMPLETE
PRIMARY:  22449 matches
BACKUP:   579 matches
FINAL:    435 matches
TOTAL:    25185 matches
UNMATCHED: 10906


In [47]:
# =====================================================================================
# ANALYSIS: TOP UNMATCHED RECRUITS BY RECRUITING RANK
# =====================================================================================

print("\n" + "="*80)
print("TOP UNMATCHED RECRUITS BY RECRUITING RANK")
print("="*80)

# Load unmatched recruits with full data
unmatched_file = os.path.join(output_dir, "unmatched_recruits.csv")
unmatched_full = pd.read_csv(unmatched_file)

print(f"\nTotal unmatched recruits: {len(unmatched_full)}")

# Merge with original recruits data to get rating information
unmatched_with_rating = unmatched_full.merge(
    recruits_df[['player_id', 'rating', 'name', 'year', 'position', 'committed_to']],
    left_on='recruit_id',
    right_on='player_id',
    how='left'
)

#Filter out recruit_year >=2026
unmatched_with_rating = unmatched_with_rating[unmatched_with_rating['year'] < 2026]

# Sort by rating (higher ratings = better recruits, assuming numeric scale)
# First, convert rating to numeric, handling any non-numeric values
unmatched_with_rating['rating_numeric'] = pd.to_numeric(unmatched_with_rating['rating'], errors='coerce')

# Sort by rating descending (best recruits first)
top_unmatched = unmatched_with_rating.sort_values('rating_numeric', ascending=False, na_position='last')

print(f"\nTop 30 Unmatched Recruits by Recruiting Rank:\n")

display_cols = ['name', 'year', 'position', 'committed_to', 'rating', 'reason']
available_cols = [c for c in display_cols if c in top_unmatched.columns]

# Show top 30
print(top_unmatched[available_cols].head(30).to_string(index=False))

print(f"\n\nReason Breakdown for Top 100 Unmatched:")
top_100_unmatched = top_unmatched.head(100)
print(top_100_unmatched['reason'].value_counts().to_string())



TOP UNMATCHED RECRUITS BY RECRUITING RANK

Total unmatched recruits: 10906

Top 30 Unmatched Recruits by Recruiting Rank:

             name  year position  rating          reason
     Aaron Wilson  2022       DL  0.9117   no_candidates
   Robby Snelling  2022       LB  0.9000   no_candidates
       Sam Thomas  2024      IOL  0.9000   no_candidates
  Gabriel Douglas  2018       WR  0.8982   no_candidates
  Trevonte Rucker  2021       WR  0.8978   no_candidates
     Brevin White  2018      PRO  0.8946   no_candidates
      Jalen Suggs  2020     DUAL  0.8933   no_candidates
Treyveon Longmire  2022      ATH  0.8926   no_candidates
 Lamar Washington  2022       LB  0.8904   no_candidates
   Landon Dulaney  2025       WR  0.8900   no_candidates
      Tiyon Evans  2019       RB  0.8900 ambiguous_match
    Javier Morton  2020        S  0.8897   no_candidates
      Jason Bargy  2019      SDE  0.8893   no_candidates
  Laquan Robinson  2022        S  0.8859   no_candidates
  Javonte Gardner  20